# 📊 Unveiling Social Media Trends
## A Comprehensive Data Exploration and Preprocessing Analysis
---
**Project Goals:**
* Generate a high-fidelity synthetic social media dataset.
* Perform rigorous data cleaning (missing values, duplicates, outliers).
* Feature engineering through binning and string manipulation.
* Visualize engagement patterns using distribution and outlier analysis.

### 🛠️ 1. Data Generation
We start by simulating a dataset of 1,000 posts including user demographics, engagement metrics (Likes/Retweets), and metadata.

In [ ]:
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

def generate_social_media_data(num_posts=1000):
    data = []
    for post_id in range(1, num_posts + 1):
        post_date = datetime(2023, 1, 1) + timedelta(days=random.randint(0, 364))
        data.append({
            'PostID': post_id,
            'UserID': random.randint(1001, 2000),
            'PostContent': "This is a synthetic post. #example",
            'PostDate': post_date,
            'UserAge': random.randint(18, 60),
            'UserGender': random.choice(['Male', 'Female', 'Other']),
            'UserLocation': random.choice(['Urban', 'Suburban', 'Rural']),
            'Likes': random.randint(0, 500),
            'Retweets': random.randint(0, 200),
            'Hashtags': random.choice(['#example', '#trending', '#news', '#tech', '#social', np.nan]),
            'Sentiment': random.choice(['Positive', 'Negative', 'Neutral', np.nan]),
            'VerifiedUser': random.choice([True, False])
        })
    return pd.DataFrame(data)

df = generate_social_media_data()
df['PostDate'] = pd.to_datetime(df['PostDate'])
print(f"Dataset Shape: {df.shape}")
df.head()

### 🧹 2. Data Cleaning & Missing Value Handling
Before analysis, we must handle null values. 

**Strategy:**
1. **Sentiment:** Impute missing values with the **Mode**.
2. **Duplicates:** Remove any redundant rows.

In [ ]:
# Missing Value Identification
print("Missing Values:\n", df.isnull().sum())

# Targeted Imputation
mode_sentiment = df['Sentiment'].mode()[0]
df['Sentiment'] = df['Sentiment'].fillna(mode_sentiment)
df['Hashtags'] = df['Hashtags'].fillna('no hashtags')

# Column Renaming & Type Conversion
df.rename(columns={'PostID': 'Tweet_ID', 'UserID': 'User_ID'}, inplace=True)
df = df.drop_duplicates()

print("\nCleaning Complete.")

### 📈 3. Feature Engineering & Binning
We create categorical segments from numerical data to find behavioral patterns.

In [ ]:
# Age Group Binning
bins = [18, 25, 35, 45, 100]
labels = ['18-25', '26-35', '36-45', '46+']
df['AgeGroup'] = pd.cut(df['UserAge'], bins=bins, labels=labels, right=True)

# Engagement Level Binning (Quantile-based)
df['EngagementCategory'] = pd.qcut(df['Likes'], 4, labels=['Low', 'Medium-Low', 'Medium-High', 'High'])

df[['UserAge', 'AgeGroup', 'Likes', 'EngagementCategory']].head()

### 🔍 4. Outlier Detection
Using the **Interquartile Range (IQR)** method to identify anomalies:
$$IQR = Q3 - Q1$$
$$Boundaries = [Q1 - 1.5 \times IQR, Q3 + 1.5 \times IQR]$$

In [ ]:
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    return df[~((df[column] < (Q1 - 1.5 * IQR)) | (df[column] > (Q3 + 1.5 * IQR)))]

df_cleaned = remove_outliers(df, 'Likes')
print(f"Removed {len(df) - len(df_cleaned)} outliers.")

### 🎨 5. Visual Data Discovery
Visualizing the distribution of users and engagement correlations.

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Age Distribution
sns.countplot(x='AgeGroup', data=df, palette="viridis", ax=axes[0])
axes[0].set_title('User Distribution by Age Group')

# Plot 2: Retweets vs Engagement Category
sns.boxplot(x='EngagementCategory', y='Retweets', data=df, palette="coolwarm", ax=axes[1])
axes[1].set_title('Engagement Category vs Retweets')

plt.tight_layout()
plt.show()